In [4]:
import os, re, time, zipfile, random
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlsplit
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

URL = "https://imgur.com/gallery/sometimes-you-just-need-cat-pictures-rd2R12A"
EXPECTED_COUNT = 22
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".gif"}
SAVE_DIR = "cat_pictures"
ZIP_NAME = "cat_pictures.zip"

In [5]:
# UA
session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/128.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://imgur.com/"
})
retry = Retry(
    total=5,
    connect=3,
    read=3,
    status=5,
    status_forcelist=[429, 500, 502, 503, 504],
    backoff_factor=1.2,
    respect_retry_after_header=True
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

resp = session.get(URL, timeout=30)
resp.raise_for_status()
html = resp.text

In [6]:
#  i.imgur.com
pattern = re.compile(
    r"https?://i\.imgur\.com/([A-Za-z0-9]{5,15})([a-z])?\.(jpg|jpeg|png|gif)",
    re.IGNORECASE
)
matches = pattern.findall(html)

seen_ids = set()
ordered_original_urls = []
for file_id, _size, ext in matches:
    if file_id in seen_ids:
        continue
    seen_ids.add(file_id)
    ordered_original_urls.append(f"https://i.imgur.com/{file_id}.{ext.lower()}")

print(f"Found {len(ordered_original_urls)} image URLs on page.")
if len(ordered_original_urls) < EXPECTED_COUNT:
    print(f" Less than expected ({EXPECTED_COUNT}). Continue anyway...")

Found 23 image URLs on page.


In [7]:
# download
def download_images(urls, save_dir, sess, max_retries=3, timeout=30):
    os.makedirs(save_dir, exist_ok=True)
    saved_files = []

    def _download_one(url):
        name = os.path.basename(urlsplit(url).path) or f"{int(time.time()*1000)}.jpg"
        fpath = os.path.join(save_dir, name)
        if os.path.exists(fpath):
            return fpath, "skip"
        for i in range(max_retries):
            try:
                with sess.get(url, stream=True, timeout=timeout) as r:
                    r.raise_for_status()
                    with open(fpath, "wb") as f:
                        for chunk in r.iter_content(64 * 1024):
                            if chunk:
                                f.write(chunk)
                return fpath, "ok"
            except Exception as e:
                if i == max_retries - 1:
                    return None, f"fail:{e}"


    for idx, u in enumerate(urls, 1):
        fpath, status = _download_one(u)
        if status == "ok":
            saved_files.append(fpath)
            print(f"[{idx}/{len(urls)}] downloaded → {fpath}")
        elif status == "skip":
            print(f"[{idx}/{len(urls)}] exists, skip → {fpath}")
        else:
            print(f"[{idx}/{len(urls)}] failed → {u}")


    return saved_files

saved = download_images(ordered_original_urls, SAVE_DIR, session)
print(f"Have downloaded: {len(saved)} images (out of {len(ordered_original_urls)} found)")


[1/23] exists, skip → cat_pictures/zWI87WCh.jpg
[2/23] exists, skip → cat_pictures/zWI87WC.jpeg
[3/23] exists, skip → cat_pictures/pVwkDoe.jpeg
[4/23] exists, skip → cat_pictures/9jHA0l7.jpeg
[5/23] exists, skip → cat_pictures/vCFKrec.jpeg
[6/23] exists, skip → cat_pictures/827D2SL.jpeg
[7/23] exists, skip → cat_pictures/l6ZXwhD.jpeg
[8/23] exists, skip → cat_pictures/8h3JdJf.jpeg
[9/23] exists, skip → cat_pictures/DB6tpis.jpeg
[10/23] exists, skip → cat_pictures/8dQz2zn.jpeg
[11/23] exists, skip → cat_pictures/XAN3LCF.jpeg
[12/23] exists, skip → cat_pictures/6Fo6UWj.jpeg
[13/23] exists, skip → cat_pictures/ks8jlE1.jpeg
[14/23] exists, skip → cat_pictures/5w0ZVXv.jpeg
[15/23] exists, skip → cat_pictures/rR3B00U.jpeg
[16/23] exists, skip → cat_pictures/41PvIwE.jpeg
[17/23] exists, skip → cat_pictures/pOUyDY8.jpeg
[18/23] exists, skip → cat_pictures/eOqoShv.jpeg
[19/23] exists, skip → cat_pictures/1h8XEtq.jpeg
[20/23] exists, skip → cat_pictures/BU6VHtN.jpeg
[21/23] exists, skip → cat_pi

In [8]:
#ZIP
def zip_folder_images(folder, zip_name="images.zip"):
    zip_path = os.path.abspath(zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(folder):
            for fn in files:
                if os.path.splitext(fn)[1].lower() in IMAGE_EXTS:
                    full = os.path.join(root, fn)
                    zf.write(full, arcname=os.path.relpath(full, start=folder))
    return zip_path

zip_path = zip_folder_images(SAVE_DIR, ZIP_NAME)
print("ZIP created:", zip_path)

ZIP created: /content/cat_pictures.zip
